In [1]:
#import libs
import scicone
import numpy as np
import pickle
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import io
import scanpy as sc
import anndata
import pyranges
import gseapy as gp
import json

In [2]:
install_path = '/cluster/work/bewi/members/andress/SCICoNE_lab/build/'
local_path ='/home/andress/pylabs/SCICoNE_lab/build'
temporary_outpath = './'
adatas_path = '/home/andress/pylabs/SCICoNE_lab/rna_imp/adatas/'
seed = 42 
np.random.seed(seed)

# Create SCICoNE object
sci = scicone.SCICoNE(local_path, temporary_outpath, verbose=False)

Using binaries at /home/andress/pylabs/SCICoNE_lab/build


In [3]:
mean_clusters = anndata.read_h5ad(f'{adatas_path}clusters_mean.h5ad')
median_clusters = anndata.read_h5ad(f'{adatas_path}clusters_median.h5ad')
sum_clusters = anndata.read_h5ad(f'{adatas_path}clusters_sum.h5ad')
mean_clusters_transformed = anndata.read_h5ad(f'{adatas_path}clusters_mean_transformed.h5ad')
median_clusters_transformed = anndata.read_h5ad(f'{adatas_path}clusters_median_transformed.h5ad')
sum_clusters_transformed = anndata.read_h5ad(f'{adatas_path}clusters_sum_transformed.h5ad')

In [ ]:
#print shape of all adatas
print("Shape of mean_clusters:", mean_clusters.shape)
print("Shape of median_clusters:", median_clusters.shape)
print("Shape of sum_clusters:", sum_clusters.shape)
print("Shape of mean_clusters_transformed:", mean_clusters_transformed.shape)
print("Shape of median_clusters_transformed:", median_clusters_transformed.shape)
print("Shape of sum_clusters_transformed:", sum_clusters_transformed.shape)


Shape of mean_clusters: (52, 17153)
Shape of median_clusters: (52, 17153)
Shape of sum_clusters: (52, 17153)
Shape of mean_clusters_transformed: (52, 17153)
Shape of median_clusters_transformed: (52, 17153)
Shape of sum_clusters_transformed: (52, 17153)


In [7]:
adata = anndata.read_h5ad('/home/andress/pylabs/SCICoNE_lab/rna_imp/adatas/adata_leiden.h5ad') 


gr_annotations = pd.read_csv('/home/andress/pylabs/SCICoNE_lab/rna_imp/gr_annotations.csv')

# Merge Chromosome and End information into adata.var
# adata.var = adata.var.merge(gr_annotations_df[['ensembl_gene_id', 'Chromosome', 'End']], 
#                             on='ensembl_gene_id', how='left')

# print(adata.var.columns)

# adata.var['Chromosome'] = adata.var['Chromosome'].astype(str)  # Convert Chromosome to string

# # Sort adata.var by Chromosome and End
# sorted_var = adata.var.sort_values(by=['Chromosome', 'End'])

# # Reorder adata.X columns based on the sorted var index
# adata = adata[:, sorted_var.index]

df_annotations = gr_annotations.set_index('ensembl_gene_id')

df_exp_annotations = df_annotations.loc[df_annotations.index.intersection(adata.var['ensembl_gene_id'])]\
                        .reset_index().rename(columns={'index':'ensembl_gene_id'})

adata.var_names = adata.var['ensembl_gene_id'].values

adata = adata[:,df_exp_annotations['ensembl_gene_id']]


df_exp_annotations_sorted = df_exp_annotations.sort_values(by=['Chromosome', 'End'])



chr_var_names = dict()
for chromosome in df_exp_annotations_sorted['Chromosome'].unique():
    chr_var_names[chromosome] = df_exp_annotations_sorted.query(f' Chromosome=="{chromosome}" ')\
                                .sort_values('Start')['ensembl_gene_id'].values

# Load the JSON file as a dictionary
json_file_path = '/home/andress/pylabs/SCICoNE_lab/rna_imp/chromosome_stops.json'
with open(json_file_path, 'r') as file:
    chromosome_stops = json.load(file)

#sort the dictionary by values
chromosome_stops = {k: v for k, v in sorted(chromosome_stops.items(), key=lambda item: item[1])}
chromosome_stops = {np.str_(k): np.int64(v) for k, v in chromosome_stops.items()}

# Use chromosome_stops directly without further processing
chromosome_stops_list = sorted(list(chromosome_stops.values()))
print(chromosome_stops_list)

#print the keys of the dictionary and the values
for key, value in chromosome_stops.items():
    print(f"Key: {key}, Value: {value}")

[np.int64(0), np.int64(218), np.int64(2859), np.int64(3508), np.int64(4457), np.int64(5335), np.int64(6090), np.int64(6370), np.int64(7179), np.int64(8215), np.int64(8612), np.int64(9783), np.int64(10119), np.int64(11024), np.int64(11571), np.int64(12091), np.int64(12519), np.int64(13423), np.int64(14172), np.int64(15474), np.int64(15845), np.int64(16110), np.int64(16505), np.int64(16700), np.int64(17151), np.int64(17152)]
Key: Start, Value: 0
Key: 1, Value: 218
Key: 2, Value: 2859
Key: 3, Value: 3508
Key: 4, Value: 4457
Key: 5, Value: 5335
Key: 6, Value: 6090
Key: 7, Value: 6370
Key: 8, Value: 7179
Key: 9, Value: 8215
Key: 10, Value: 8612
Key: 11, Value: 9783
Key: 12, Value: 10119
Key: 13, Value: 11024
Key: 14, Value: 11571
Key: 15, Value: 12091
Key: 16, Value: 12519
Key: 17, Value: 13423
Key: 18, Value: 14172
Key: 19, Value: 15474
Key: 20, Value: 15845
Key: 21, Value: 16110
Key: 22, Value: 16505
Key: X, Value: 16700
Key: Y, Value: 17151
Key: End, Value: 17152


In [9]:
print(f"Type of filtered_chromosome_stops: {type(chromosome_stops)}")
print(f"Content of filtered_chromosome_stops:\n{chromosome_stops}")

# Print shape of filtered_counts
#print(f"Shape of filtered_counts: {chromosome_stops.shape}")
# Print shape of filtered_chromosome_stops
print(f"Shape of filtered_chromosome_stops: {len(chromosome_stops)}")

Type of filtered_chromosome_stops: <class 'dict'>
Content of filtered_chromosome_stops:
{np.str_('Start'): np.int64(0), np.str_('1'): np.int64(218), np.str_('2'): np.int64(2859), np.str_('3'): np.int64(3508), np.str_('4'): np.int64(4457), np.str_('5'): np.int64(5335), np.str_('6'): np.int64(6090), np.str_('7'): np.int64(6370), np.str_('8'): np.int64(7179), np.str_('9'): np.int64(8215), np.str_('10'): np.int64(8612), np.str_('11'): np.int64(9783), np.str_('12'): np.int64(10119), np.str_('13'): np.int64(11024), np.str_('14'): np.int64(11571), np.str_('15'): np.int64(12091), np.str_('16'): np.int64(12519), np.str_('17'): np.int64(13423), np.str_('18'): np.int64(14172), np.str_('19'): np.int64(15474), np.str_('20'): np.int64(15845), np.str_('21'): np.int64(16110), np.str_('22'): np.int64(16505), np.str_('X'): np.int64(16700), np.str_('Y'): np.int64(17151), np.str_('End'): np.int64(17152)}
Shape of filtered_chromosome_stops: 26


In [4]:
mean_clusters = anndata.read_h5ad('/home/andress/pylabs/SCICoNE_lab/rna_imp/adatas/clusters_mean.h5ad')


print(type(mean_clusters.X))  # Check the type
print(mean_clusters.X.shape)  # Check the shape
print(mean_clusters.X[:5, :5])  # Inspect the first 5 rows and columns
print(mean_clusters.X.dtype)  # Check the data type

mean_clusters

<class 'numpy.ndarray'>
(52, 17153)
[[0.         0.         0.         0.04166667 0.04166667]
 [0.03333333 0.         0.         0.         0.        ]
 [0.20588235 0.         0.02941176 0.11764706 0.05882353]
 [0.10909091 0.01818182 0.01818182 0.01818182 0.01818182]
 [0.1147541  0.         0.         0.03278689 0.01639344]]
float64


AnnData object with n_obs × n_vars = 52 × 17153
    obs: 'cluster'
    var: 'ensembl_gene_id', 'gene_id', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'Chromosome', 'End'

In [6]:
# Ensure 'Chromosome' and 'End' columns exist in mean_clusters.var
if 'Chromosome' in mean_clusters.var.columns and 'End' in mean_clusters.var.columns:
    # Filter for Chromosome 1 and extract all End values
    chrom1_ends = mean_clusters.var[mean_clusters.var['Chromosome'] == '1']['End']
    print("All ending positions for Chromosome 1:")
    print(chrom1_ends.tolist())  # Convert to list for easier readability
else:
    print("Chromosome or End column not found in mean_clusters.var")

All ending positions for Chromosome 1:
[259024, 485208, 774280, 810066, 827989, 877032, 921034, 959309, 965719, 998051, 1014540, 1056119, 1074306, 1116361, 1170343, 1197936, 1206592, 1214153, 1232031, 1235041, 1246722, 1273864, 1280420, 1292029, 1309609, 1311677, 1324687, 1328896, 1349418, 1361777, 1375207, 1399335, 1402054, 1407293, 1410618, 1421769, 1442882, 1470163, 1497848, 1534685, 1574863, 1577075, 1600135, 1615795, 1630605, 1635263, 1659012, 1662939, 1674397, 1692795, 1724357, 1739557, 1780457, 1892292, 1892835, 2003837, 2019198, 2145279, 2184389, 2185395, 2212720, 2310213, 2391707, 2405442, 2413797, 2505532, 2526597, 2565382, 2591469, 2633016, 2801693, 3481113, 3624787, 3630127, 3652761, 3736201, 3746742, 3775982, 3796498, 3857396, 3885429, 3900293, 3917547, 6180321, 6209389, 6221299, 6235972, 6239448, 6261098, 6393767, 6424670, 6461367, 6466175, 6520074, 6554513, 6589280, 6614607, 6624030, 6635586, 6701924, 6730012, 7769706, 7781432, 7845177, 7985505, 8026309, 8187014, 8344167

In [ ]:
print("Type of mean_clusters.X:", type(mean_clusters.X))
print("Shape of mean_clusters.X:", mean_clusters.X.shape)
print("First 5 rows and columns of mean_clusters.X:")
print(mean_clusters.X[:5, :5])
print("Data type of mean_clusters.X:", mean_clusters.X.dtype)

In [ ]:
#adata = anndata.read_h5ad('/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/adata_clustered_sorted.h5ad')  
adata = mean_clusters
gr_annotations = pd.read_csv('/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/gr_annotations.csv')

df_annotations = gr_annotations.set_index('ensembl_gene_id')

df_exp_annotations = df_annotations.loc[df_annotations.index.intersection(adata.var['ensembl_gene_id'])]\
                        .reset_index().rename(columns={'index':'ensembl_gene_id'})

adata.var_names = adata.var['ensembl_gene_id'].values

adata = adata[:,df_exp_annotations['ensembl_gene_id']]


df_exp_annotations_sorted = df_exp_annotations.sort_values(by=['Chromosome', 'End'])

chr_var_names = dict()
for chromosome in df_exp_annotations_sorted['Chromosome'].unique():
    chr_var_names[chromosome] = df_exp_annotations_sorted.query(f' Chromosome=="{chromosome}" ')\
                                    .sort_values('Start')['ensembl_gene_id'].values

# with open(f'{temporary_outpath}/chromosome_stops.json', 'r') as f:
#     chromosome_stops = json.load(f)


# chrom_stops_df = pd.read_csv('/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/stop.csv')
# chromosome_stops = chrom_stops_df['Chromosome_stops'].tolist()

chromosome_stops = [0, 1878, 3093, 4129, 4771, 5564, 6558, 7379, 8587, 9305, 9981, 10965, 11906, 12178, 12757, 13363, 14192, 15253, 15549, 17244, 15964, 17884, 17691, 7937, 17906]



In [ ]:
# mean_cluster = anndata.read_h5ad('/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/mean_clusters_sorted.h5ad')

# mean_clusters.var['Chromosome'] = mean_clusters.var['Chromosome'].astype(str)
# sorted_var = mean_clusters.var.sort_values(by=['Chromosome', 'End'])

# Reorder adata.X columns based on the sorted var index
# mean_clusters = mean_clusters[:, sorted_var.index]
data = mean_clusters.X
#data = np.nan_to_num(data, nan=0.0, posinf=1e9, neginf=-1e9)

#data = np.nan_to_num(data, nan=0.0, posinf=1e9, neginf=-1e9)


In [ ]:
#sci.detect_breakpoints(data=data, window_size=10, threshold=3)
sci.detect_breakpoints(data = data, window_size=200, threshold=3, input_breakpoints=chromosome_stops)


scicone.plotting.plot_matrix(data, bps=sci.bps['segmented_regions'],
                                chr_stops_dict=chromosome_stops,
                                cbar_title='Normalized\n  counts', vmax=2, cluster=False)

In [ ]:
#print shape of all adatas
